# CNN — Detecção de Células Cervicais (SipakMed)

**Dataset:** SipakMed — 5 classes de células do colo do útero  
**Modelo:** MobileNetV2 com Transfer Learning  
**Output:** `cnn_cervical.h5` para uso no Streamlit

### Pré-requisito
Faça upload do dataset SipakMed no Google Drive antes de rodar.
Estrutura esperada no Drive:
```
MyDrive/sipakmed/
  im_Dyskeratotic/im_Dyskeratotic/*.bmp
  im_Koilocytotic/im_Koilocytotic/*.bmp
  im_Metaplastic/im_Metaplastic/*.bmp
  im_Parabasal/im_Parabasal/*.bmp
  im_Superficial-Intermediate/im_Superficial-Intermediate/*.bmp
```

### Ativar GPU no Colab
**Runtime → Change runtime type → T4 GPU → Save**

## 1. Monta o Google Drive e configura os caminhos

In [ ]:
# Monta o Google Drive — vai abrir uma janela pedindo permissão
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Caminho para o SipakMed no seu Google Drive
# Ajuste se você colocou em uma subpasta diferente
SIPAKMED_PATH = '/content/drive/MyDrive/sipakmed'

# Pasta onde o modelo treinado será salvo (também no Drive para não perder)
OUTPUT_PATH = '/content/drive/MyDrive/cervical_artifacts'
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Verifica se o dataset foi encontrado
assert os.path.exists(SIPAKMED_PATH), f'SipakMed não encontrado em: {SIPAKMED_PATH}'

# Lista as classes disponíveis
classes = sorted([d for d in os.listdir(SIPAKMED_PATH)
                  if os.path.isdir(os.path.join(SIPAKMED_PATH, d))])
print(f'Classes encontradas: {classes}')
print(f'Total de classes: {len(classes)}')

## 2. Instala dependências e imports

In [ ]:
# Instala o opencv para converter BMP → array numpy
!pip install -q opencv-python-headless

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2          # Para carregar imagens BMP
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Verifica se a GPU está disponível
print('TensorFlow:', tf.__version__)
print('GPU disponível:', tf.config.list_physical_devices('GPU'))

## 3. Carrega e prepara as imagens

In [ ]:
# Configurações do modelo
IMG_SIZE = (224, 224)   # Tamanho esperado pelo MobileNetV2
BATCH_SIZE = 32         # Imagens por batch de treinamento

# Mapeamento das 5 classes para binário:
# Anômala (1): Dyskeratotic e Koilocytotic — células alteradas
# Normal  (0): Metaplastic, Parabasal, Superficial-Intermediate
BINARY_MAP = {
    'im_Dyskeratotic':           1,   # Displásica — anômala
    'im_Koilocytotic':           1,   # Coilocitose HPV — anômala
    'im_Metaplastic':            0,   # Metaplásica — normal
    'im_Parabasal':              0,   # Parabasal — normal
    'im_Superficial-Intermediate': 0, # Superficial/Intermediária — normal
}

print('Mapeamento de classes:')
for k, v in BINARY_MAP.items():
    print(f'  {k}: {"Anômala (1)" if v == 1 else "Normal (0)"}')

In [ ]:
# Carrega todas as imagens .bmp e converte para arrays numpy
# Isso pode levar alguns minutos dependendo do Drive

images = []   # Lista de arrays (224, 224, 3)
labels = []   # Lista de rótulos binários (0 ou 1)

for class_folder, label in BINARY_MAP.items():
    # O SipakMed tem subpasta com mesmo nome: im_Dyskeratotic/im_Dyskeratotic/
    class_path = os.path.join(SIPAKMED_PATH, class_folder, class_folder)

    if not os.path.isdir(class_path):
        print(f'  Pasta não encontrada: {class_path}')
        continue

    # Lista apenas arquivos .bmp (ignora .dat)
    bmp_files = [f for f in os.listdir(class_path) if f.endswith('.bmp')]
    print(f'  {class_folder}: {len(bmp_files)} imagens')

    for fname in bmp_files:
        fpath = os.path.join(class_path, fname)

        # Carrega com OpenCV (suporte nativo a BMP)
        img = cv2.imread(fpath)
        if img is None:
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)   # BGR → RGB
        img = cv2.resize(img, IMG_SIZE)               # Redimensiona para 224x224
        img = img / 255.0                             # Normaliza para [0, 1]

        images.append(img)
        labels.append(label)

# Converte para arrays numpy
X = np.array(images, dtype=np.float32)   # Shape: (n, 224, 224, 3)
y = np.array(labels, dtype=np.int32)     # Shape: (n,)

print(f'\nTotal de imagens carregadas: {len(X)}')
print(f'Shape X: {X.shape}')
print(f'Normais (0): {(y==0).sum()} | Anômalas (1): {(y==1).sum()}')

## 4. Split e Data Augmentation

In [ ]:
# Split: 70% treino, 15% validação, 15% teste
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Treino:    {len(X_train)} imagens')
print(f'Validação: {len(X_val)} imagens')
print(f'Teste:     {len(X_test)} imagens')

In [ ]:
# Data Augmentation — gera variações das imagens de treino para o modelo
# generalizar melhor e não decorar as imagens originais (overfitting)
datagen = ImageDataGenerator(
    rotation_range=20,       # Rotação aleatória até 20 graus
    width_shift_range=0.1,   # Deslocamento horizontal
    height_shift_range=0.1,  # Deslocamento vertical
    zoom_range=0.1,          # Zoom aleatório
    horizontal_flip=True,    # Espelhamento horizontal
    vertical_flip=True,      # Espelhamento vertical (células não têm orientação fixa)
    fill_mode='nearest'      # Preenche pixels novos com o vizinho mais próximo
)

# Cria o gerador de treino com augmentation
train_gen = datagen.flow(X_train, y_train, batch_size=BATCH_SIZE)

print('Data augmentation configurado!')
print(f'Steps por epoch: {len(X_train) // BATCH_SIZE}')

## 5. Constrói o modelo (MobileNetV2 + Transfer Learning)

In [ ]:
# Transfer Learning com MobileNetV2:
# - Carrega o MobileNetV2 pré-treinado no ImageNet (1.2M imagens, 1000 classes)
# - Remove a camada de classificação original
# - Congela os pesos existentes (não retreina — só aprende as camadas novas)
# - Adiciona nossas camadas de classificação binária no topo
#
# Por que MobileNetV2?
# - Leve e rápido (ideal para CPU/GPU modesta)
# - Boa performance em tarefas de visão médica
# - Transfer Learning: já sabe detectar bordas, texturas, padrões celulares

# Carrega a base do MobileNetV2 sem a camada final
base_model = MobileNetV2(
    input_shape=(224, 224, 3),  # Tamanho das imagens
    include_top=False,          # Remove a camada de classificação original
    weights='imagenet'          # Pesos pré-treinados no ImageNet
)

# Congela os pesos da base — não queremos alterar o que já foi aprendido
base_model.trainable = False

# Constrói o modelo completo
inputs = keras.Input(shape=(224, 224, 3))

# Passa pela base pré-treinada (extrator de features)
x = base_model(inputs, training=False)

# Global Average Pooling — reduz (7, 7, 1280) para (1280,)
x = layers.GlobalAveragePooling2D()(x)

# Camada densa com dropout para evitar overfitting
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)   # Desativa 30% dos neurônios aleatoriamente no treino

# Camada de saída binária: sigmoid → probabilidade entre 0 e 1
outputs = layers.Dense(1, activation='sigmoid')(x)

model = keras.Model(inputs, outputs)

# Compila o modelo
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),  # Otimizador Adam
    loss='binary_crossentropy',   # Loss para classificação binária
    metrics=['accuracy',
             keras.metrics.Recall(name='recall'),      # Recall = mais importante!
             keras.metrics.AUC(name='auc')]            # AUC-ROC
)

model.summary()

## 6. Treinamento — Fase 1 (cabeça congelada)

In [ ]:
# Callbacks — comportamentos automáticos durante o treinamento
callbacks = [
    # Para o treino se a validação não melhorar por 5 epochs consecutivas
    EarlyStopping(monitor='val_recall', patience=5, restore_best_weights=True, mode='max'),

    # Reduz o learning rate se ficar estagnado
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),

    # Salva o melhor modelo no Drive automaticamente
    ModelCheckpoint(
        filepath=os.path.join(OUTPUT_PATH, 'cnn_cervical_best.h5'),
        monitor='val_recall',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

# Calcula pesos para lidar com desbalanceamento de classes
n_normal  = (y_train == 0).sum()
n_anomal  = (y_train == 1).sum()
total     = len(y_train)
class_weight = {
    0: total / (2 * n_normal),   # Peso para classe Normal
    1: total / (2 * n_anomal)    # Peso para classe Anômala (provavelmente maior)
}
print(f'Class weights: {class_weight}')

# Treina a fase 1: apenas as camadas novas (base congelada)
print('\n=== FASE 1: Treinamento das camadas novas ===')
history1 = model.fit(
    train_gen,
    epochs=20,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    class_weight=class_weight,
    verbose=1
)

## 7. Fine-tuning — Fase 2 (descongela as últimas camadas)

In [ ]:
# Fine-tuning: descongela as últimas 30 camadas da base
# Permite que o modelo ajuste os pesos pré-treinados para nosso domínio
base_model.trainable = True

# Mantém as primeiras camadas congeladas (detectores de bordas básicos)
# Descongela apenas as últimas 30 (detectores de padrões mais complexos)
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompila com learning rate menor para não destruir os pesos aprendidos
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),  # LR muito menor
    loss='binary_crossentropy',
    metrics=['accuracy',
             keras.metrics.Recall(name='recall'),
             keras.metrics.AUC(name='auc')]
)

print('=== FASE 2: Fine-tuning das últimas 30 camadas ===')
history2 = model.fit(
    train_gen,
    epochs=20,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    class_weight=class_weight,
    verbose=1
)

## 8. Avaliação no conjunto de teste

In [ ]:
# Avalia no conjunto de teste (nunca visto durante o treino)
print('=== AVALIAÇÃO FINAL — CONJUNTO DE TESTE ===')
test_loss, test_acc, test_recall, test_auc = model.evaluate(X_test, y_test, verbose=0)
print(f'Accuracy: {test_acc:.4f}')
print(f'Recall:   {test_recall:.4f}  ← mais importante!')
print(f'AUC-ROC:  {test_auc:.4f}')

# Predições
y_pred_proba = model.predict(X_test).ravel()
y_pred = (y_pred_proba >= 0.5).astype(int)

print('\nRelatório completo:')
print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Anômala (1)']))

In [ ]:
# Matriz de confusão
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Anômala'],
            yticklabels=['Normal', 'Anômala'])
plt.title('Matriz de Confusão — CNN Cervical')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

In [ ]:
# Curvas de treinamento
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Junta histórico das duas fases
acc  = history1.history['accuracy']  + history2.history['accuracy']
vacc = history1.history['val_accuracy'] + history2.history['val_accuracy']
rec  = history1.history['recall']    + history2.history['recall']
vrec = history1.history['val_recall']   + history2.history['val_recall']

axes[0].plot(acc,  label='Treino')
axes[0].plot(vacc, label='Validação')
axes[0].set_title('Accuracy')
axes[0].legend()

axes[1].plot(rec,  label='Treino')
axes[1].plot(vrec, label='Validação')
axes[1].set_title('Recall')
axes[1].legend()

plt.suptitle('Curvas de Treinamento — Fase 1 + Fase 2', fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Salva o modelo final

In [ ]:
# Salva o modelo no Google Drive
model_path = os.path.join(OUTPUT_PATH, 'cnn_cervical.h5')
model.save(model_path)
print(f'Modelo salvo em: {model_path}')

# Tamanho do arquivo
size_mb = os.path.getsize(model_path) / (1024 * 1024)
print(f'Tamanho: {size_mb:.1f} MB')

print('\n✅ Pronto!')
print('Baixe o arquivo cnn_cervical.h5 do Drive e coloque em:')
print('  app/models/artifacts/cnn_cervical.h5')

In [ ]:
# Opção alternativa: baixar direto pelo Colab sem passar pelo Drive
from google.colab import files
files.download(model_path)